# Severstal Steel Defect Detection — CENG 476 Deep Learning Project**Görev:** Çelik yüzeyi fotoğraflarında 4 sınıflı semantic segmentation (piksel bazlı kusur tespiti)**Bu notebook'un akışı:**1. Kurulum ve veri kontrolü2. Keşifsel veri analizi (EDA) — sınıf dağılımı, örnek görüntüler3. Hızlı doğrulama (smoke test)4. Baseline model eğitimi5. Final model eğitimi6. Değerlendirme, grafikler ve sonuç tabloları7. Ablasyon karşılaştırması> Her bölümde **ne yaptığımız** kadar **neden yaptığımız** da yazıyor —> sunumdaki soru-cevap bölümü için bu açıklamaları oku.

## 1. Kurulum ve veri kontrolü

In [ ]:
!pip install -q albumentations==1.4.15 2>/dev/nullimport os, sys, torchprint("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())if torch.cuda.is_available():    print("Cihaz:", torch.cuda.get_device_name(0))

In [ ]:
# Kaggle notebook'ta veri buradadır. Lokalde çalışıyorsan bu yolu değiştir.DATA_DIR = "/kaggle/input/severstal-steel-defect-detection"print(os.listdir(DATA_DIR))print("train görüntü sayısı:", len(os.listdir(os.path.join(DATA_DIR, "train_images"))))

In [ ]:
# src/ modüllerini import edilebilir hale getirSRC = "src" if os.path.isdir("src") else "../src"SRC_DIR = os.path.abspath(SRC)sys.path.insert(0, SRC_DIR)from config import Configfrom dataset import build_dataframe, split_dataframe, SteelDatasetfrom utils import rle_to_mask, set_seedset_seed(42)

## 2. Keşifsel Veri Analizi (EDA)Modeli tasarlamadan önce veriyi tanımalıyız. Özellikle iki şeyi arıyoruz:**sınıf dengesizliği** ve **kusur boyutları** — bunlar kayıp fonksiyonu vemetrik seçimimizi doğrudan belirleyecek.

In [ ]:
df = build_dataframe(DATA_DIR)print("Toplam görüntü:", len(df))print("\nHiç kusuru olmayan görüntü:", int((df["n_defects"] == 0).sum()),      f"({(df['n_defects'] == 0).mean()*100:.1f}%)")print("Birden fazla kusur sınıfı içeren:", int((df["n_defects"] > 1).sum()))df.head()

In [ ]:
import matplotlib.pyplot as pltimport numpy as npfig, axes = plt.subplots(1, 3, figsize=(16, 4))counts = [int(df[f"has{c}"].sum()) for c in range(1, 5)]axes[0].bar([f"Class {c}" for c in range(1, 5)], counts, color="tab:blue")axes[0].set_title("Sınıf başına görüntü sayısı")for i, v in enumerate(counts):    axes[0].text(i, v, str(v), ha="center", va="bottom")df["n_defects"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="tab:orange")axes[1].set_title("Görüntü başına kusur sınıfı sayısı"); axes[1].set_xlabel("kusur sınıfı sayısı")# Kusur alanlarının dağılımı (min_size filtresi için fikir verir)areas = {c: [] for c in range(1, 5)}sample = df[df["n_defects"] > 0].sample(400, random_state=42)for _, row in sample.iterrows():    for c in range(1, 5):        if isinstance(row[f"rle{c}"], str):            areas[c].append(rle_to_mask(row[f"rle{c}"]).sum())for c in range(1, 5):    if areas[c]:        axes[2].hist(np.log10(np.array(areas[c]) + 1), bins=30, alpha=0.55, label=f"Class {c}")axes[2].set_title("Kusur alanı dağılımı (log10 piksel)"); axes[2].legend()plt.tight_layout(); plt.show()

**Yorum (rapora yaz):**- Sınıflar ciddi biçimde dengesiz — Class 3 baskın, Class 2 çok nadir.  Bu yüzden **accuracy anlamsız**, Dice/IoU kullanıyoruz ve BCE'ye `pos_weight` veriyoruz.- Görüntülerin yaklaşık yarısında hiç kusur yok → yardımcı sınıflandırma başlığı  (çok görevli öğrenme) ve `min_size` son işlemi bu yüzden mantıklı.- Kusur alanları geniş bir aralığa yayılıyor; çok küçük bileşenler çoğunlukla  gürültü, `min_size` eşiğini bu histograma bakarak seçiyoruz.

In [ ]:
import cv2# Her sınıftan birer örnek: görüntü + maske overlaycolors = {1: (255, 0, 0), 2: (0, 255, 0), 3: (0, 128, 255), 4: (255, 255, 0)}fig, axes = plt.subplots(4, 1, figsize=(16, 10))for c in range(1, 5):    row = df[df[f"has{c}"] == 1].iloc[0]    img = cv2.cvtColor(cv2.imread(os.path.join(DATA_DIR, "train_images", row["ImageId"])), cv2.COLOR_BGR2RGB)    m = rle_to_mask(row[f"rle{c}"])    vis = img.copy()    vis[m > 0] = (0.45 * vis[m > 0] + 0.55 * np.array(colors[c])).astype(np.uint8)    axes[c-1].imshow(vis); axes[c-1].axis("off")    axes[c-1].set_title(f"Class {c} — {row['ImageId']} (kusur alanı: {int(m.sum())} px)", fontsize=10)plt.tight_layout(); plt.show()

In [ ]:
# Train / validation / test bölmesi — stratified (kusur sınıfı imzasına göre)train_df, val_df, test_df = split_dataframe(df, val_size=0.15, test_size=0.15, seed=42)print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")import pandas as pdpd.DataFrame({    split: [d[f"has{c}"].mean().round(4) for c in range(1, 5)] + [(d["n_defects"] == 0).mean().round(4)]    for split, d in [("train", train_df), ("val", val_df), ("test", test_df)]}, index=[f"class{c} oranı" for c in range(1, 5)] + ["kusursuz oranı"])

Üç bölmede de sınıf oranları neredeyse aynı → stratified split çalışıyor. Bu, karşılaştırmaların adil olması için şart.

## 3. Model mimarisiU-Net + ImageNet ön eğitimli ResNet34 encoder. Detaylı gerekçe `src/model.py`dosyasının başındaki açıklamada.

In [ ]:
from model import build_model, count_parameterscfg = Config(); cfg.data_dir = DATA_DIRmodel = build_model(cfg)x = torch.randn(2, 3, 256, 512)seg, cls = model(x)print("girdi:", tuple(x.shape))print("segmentasyon çıktısı:", tuple(seg.shape), "(logit — sigmoid uygulanmadı)")print("sınıflandırma çıktısı:", tuple(cls.shape))t, tr = count_parameters(model)print(f"parametre: {t/1e6:.2f}M (eğitilebilir {tr/1e6:.2f}M)")

## 4. Hızlı doğrulama (smoke test)Uzun eğitime başlamadan önce her şeyin uçtan uca çalıştığını doğrula. ~2-3 dakika.

In [ ]:
!python "{SRC_DIR}/train.py" --data_dir "{DATA_DIR}" --limit 300 --epochs 2 --run_name smoke_test

## 5. Baseline model**Hipotez:** Ön eğitim, BatchNorm, dropout, augmentasyon ve weight decay'inher biri performansa katkı sağlıyor. Bunu göstermek için hepsini **kapalı**bir referans model eğitiyoruz.

In [ ]:
!python "{SRC_DIR}/train.py" --data_dir "{DATA_DIR}" --run_name baseline \    --pretrained 0 --use_bn 0 --decoder_dropout 0.0 --use_attention 0 \    --weight_decay 0.0 --augment 0 --cls_loss_weight 0.0 --scheduler none --epochs 15

## 6. Final modelTüm teknikler açık: ön eğitimli encoder, BN, Dropout2d(0.2), SCSE attention,augmentasyon, AdamW + weight decay 1e-4, cosine annealing, early stopping (patience 6),çok görevli kayıp.

In [ ]:
!python "{SRC_DIR}/train.py" --data_dir "{DATA_DIR}" --run_name final --epochs 30

In [ ]:
from IPython.display import Image, displaydisplay(Image("outputs/final/curves.png"))

**Overfitting / underfitting analizi (rapora yaz):**Yukarıdaki grafikte train ve validation eğrileri arasındaki açıklığa bak.- Açıklık büyük ve val loss yükseliyorsa → **overfitting**. Çözümümüz: dropout artır,  weight decay artır, augmentasyon güçlendir, early stopping.- Her iki eğri de yüksek loss'ta takılı kaldıysa → **underfitting**. Çözüm: LR artır,  daha uzun eğit, model kapasitesini büyüt (ResNet34 → ResNet50).Kırmızı kesikli çizgi early stopping'in seçtiği en iyi epoch'u gösteriyor.

## 7. Değerlendirme`evaluate.py` şunları yapar: validation üzerinde en iyi (threshold, min_size)çiftini arar, flip TTA uygular, test setinde final metrikleri hesaplar,karışıklık matrisi ve örnek tahminleri çizer.> **Önemli:** Eşik ayarını validation'da yapıp test'te sabit kullanıyoruz.> Test setinde ayar yapmak veri sızıntısı olurdu.

In [ ]:
!python "{SRC_DIR}/evaluate.py" --data_dir "{DATA_DIR}" --run_name final!python "{SRC_DIR}/evaluate.py" --data_dir "{DATA_DIR}" --run_name baseline

In [ ]:
display(Image("outputs/final/confusion_matrix.png"))display(Image("outputs/final/predictions.png"))

In [ ]:
import jsonfor run in ["baseline", "final"]:    m = json.load(open(f"outputs/{run}/metrics_test.json"))    print(f"--- {run} ---  mean dice: {m['mean_dice']:.4f} | mean IoU: {m['mean_iou']:.4f} "          f"| macro F1: {m.get('macro_f1', float('nan')):.4f}")print("\nTrivial baseline (hep boş tahmin):",      round(json.load(open("outputs/final/metrics_test.json"))["baseline_all_empty_dice"], 4))

In [ ]:
# Son işlem (threshold + min_size) ayarının etkisiimport pandas as pdgrid = pd.read_csv("outputs/final/postprocess_grid.csv")pivot = grid.pivot(index="min_size", columns="threshold", values="val_dice")display(pivot.style.background_gradient(cmap="Greens").format("{:.4f}"))

## 8. Ablasyon karşılaştırmasıTüm koşuları tek tabloda topla. Bu tablo doğrudan rapordaki"Experiments and Results / 5.2" bölümüne gider.

In [ ]:
!python "{SRC_DIR}/compare_runs.py" --out_dir outputs

## 9. Sonuç ve çıkarımlar[Buraya kendi sayılarınla doldur:]- Final model test Dice: **[...]** vs baseline **[...]** vs trivial baseline **[...]**- En çok katkı sağlayan bileşen: **[...]**- En zor sınıf: **[...]** — nedeni: [...]- Kısıtlar: gerçek yarışma test etiketleri yok, sınırlı GPU saati, tek seed- Gelecek çalışma: ResNet50/EfficientNet encoder, k-fold ensemble, pseudo-labeling